<a href="https://colab.research.google.com/github/mdsium161-afk/HiveMind-Group-3/blob/main/Week4_Lab_Smart_Vitals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 Lab: Cleaning a Smart Vitals Data Stream
**CMS22301 Intelligent Systems and Emerging Technologies**

**Scenario.** A wearable patch sends a heart-rate and blood-oxygen (SpO2) reading every minute to a cloud server.
Real wearables are messy: packets get lost, sensors glitch, and straps slip. Your job is to decide **what the device should clean itself (edge)** and **what should wait for the cloud**, then check how well you did.

Run the cells **in order** (Shift+Enter). Do not skip cells.

## Step 1. Create the data
This cell builds a *clean* signal (the truth) and a *noisy* copy (what the wearable really sends). Because the random seed is fixed, **everyone gets exactly the same data**. Do not change the seed.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(7)
n = 240                                    # 240 minutes = 4 hours
times = pd.date_range("2026-10-12 08:00", periods=n, freq="min")

# TRUTH: what the patient's body is really doing
activity = np.zeros(n)
activity[60:90] = 25                       # a walk at 09:00-09:30
activity[170:190] = 30                     # climbing stairs at 10:50-11:10
true_hr = 70 + activity + np.cumsum(np.random.normal(0, 0.3, n))
true_spo2 = 97 - activity * 0.02 + np.random.normal(0, 0.2, n)
truth = pd.DataFrame({"timestamp": times, "heart_rate": true_hr, "spo2": true_spo2})

# NOISY: what the wearable actually sends
noisy = truth.copy()
drop = np.random.choice(noisy.index, size=24, replace=False)       # 10% packets lost
noisy = noisy.drop(index=drop)
glitch_idx = np.random.choice(noisy.index, size=4, replace=False)  # sensor glitches
noisy.loc[glitch_idx[:2], "heart_rate"] = [250, 5]
noisy.loc[glitch_idx[2:], "spo2"] = [30, 140]
miss_idx = np.random.choice(noisy.index, size=8, replace=False)    # SpO2 sensor blanks
noisy.loc[miss_idx, "spo2"] = np.nan
noisy["heart_rate"] = noisy["heart_rate"] + np.linspace(0, 8, len(noisy))  # strap slowly loosens (drift)
noisy = noisy.reset_index(drop=True)

print("Rows sent:", len(noisy), "of", n, "expected")
noisy.head()

Rows sent: 216 of 240 expected


,timestamp,heart_rate,spo2
0,2026-10-12 08:02:00,70.377223,97.255072
1,2026-10-12 08:03:00,70.536687,97.003872
2,2026-10-12 08:04:00,70.337219,97.147276
3,2026-10-12 08:06:00,70.374781,97.257641
4,2026-10-12 08:07:00,69.885573,96.882347


## Step 2. Inspect the problems
Before cleaning anything, **find out what is wrong**. Never clean blind.

In [2]:
print("Missing values per column:")
print(noisy.isna().sum())
print()
print("Summary statistics:")
print(noisy[["heart_rate", "spo2"]].describe().round(1))

Missing values per column:
timestamp     0
heart_rate    0
spo2          8
dtype: int64

Summary statistics:
       heart_rate   spo2
count       216.0  208.0
mean         79.2   96.8
std          16.7    5.5
min          10.5   30.0
25%          72.3   96.7
50%          74.4   96.9
75%          75.9   97.1
max         250.8  140.0


In [3]:
# How many minutes never arrived? The schedule says: one reading per minute, 08:00 to 11:59.
expected = pd.date_range("2026-10-12 08:00", periods=240, freq="min")
missing_minutes = expected.difference(noisy["timestamp"])
print("Missing minutes (lost packets):", len(missing_minutes))

# Readings that are physically impossible
bad_hr = noisy[(noisy["heart_rate"] < 30) | (noisy["heart_rate"] > 220)]
bad_spo2 = noisy[(noisy["spo2"] < 70) | (noisy["spo2"] > 100)]
print("Impossible heart-rate readings:", len(bad_hr))
print("Impossible SpO2 readings:", len(bad_spo2))

Missing minutes (lost packets): 24
Impossible heart-rate readings: 2
Impossible SpO2 readings: 2


### Your turn (write your answers in a text cell)
1. Which problem is the **most serious** for a patient-monitoring system, and why?
2. Which of these problems could the wearable fix by itself, with no history and very little power?

## Step 3. EDGE-stage cleaning (runs on the wearable)
The device has a tiny battery and almost no memory, so it can only do **cheap, instant checks**:
* **Range check:** throw away values that are impossible for a human.
* **Short gap fill:** if only 1-2 minutes are missing, repeat the last good value.

In [4]:
edge = noisy.copy()

# 1. Range check: impossible values become "missing"
edge.loc[(edge["heart_rate"] < 30) | (edge["heart_rate"] > 220), "heart_rate"] = np.nan
edge.loc[(edge["spo2"] < 70) | (edge["spo2"] > 100), "spo2"] = np.nan

# 2. Put the lost minutes back as empty rows so gaps are visible
edge = edge.set_index("timestamp").reindex(expected)
edge.index.name = "timestamp"

# 3. Short gap fill: carry the last value forward, but only up to 2 minutes
edge[["heart_rate", "spo2"]] = edge[["heart_rate", "spo2"]].ffill(limit=2)

print("Still missing after EDGE cleaning:")
print(edge[["heart_rate", "spo2"]].isna().sum())

Still missing after EDGE cleaning:
heart_rate    2
spo2          3
dtype: int64


## Step 4. CLOUD-stage cleaning (runs on the server)
The server has the whole history and plenty of power, so it can do heavier work:
* **Fill longer gaps** using a straight line between the known values before and after (interpolation).
* **Correct the drift:** the loose strap slowly pushes heart rate upwards. We estimate the drift and remove it.

In [5]:
cloud = edge.copy()

# 1. Fill the remaining (longer) gaps by interpolation
cloud[["heart_rate", "spo2"]] = cloud[["heart_rate", "spo2"]].interpolate(limit_direction="both")

# 2. Drift correction: the first 10 readings define the "baseline"; later readings
#    drift away from it, so subtract the slow trend (a 60-minute rolling median minus the baseline)
baseline = cloud["heart_rate"].iloc[:10].median()
trend = cloud["heart_rate"].rolling(60, min_periods=10).median()
cloud["heart_rate_fixed"] = cloud["heart_rate"] - (trend - baseline).fillna(0)

print("Missing after CLOUD cleaning:", int(cloud[["heart_rate", "spo2"]].isna().sum().sum()))
print("Baseline heart rate:", round(baseline, 1))
cloud.head()

Missing after CLOUD cleaning: 0
Baseline heart rate: 70.4


,heart_rate,spo2,heart_rate_fixed
timestamp,,,
2026-10-12 08:00:00,70.377223,97.255072,70.377223
2026-10-12 08:01:00,70.377223,97.255072,70.377223
2026-10-12 08:02:00,70.377223,97.255072,70.377223
2026-10-12 08:03:00,70.536687,97.003872,70.536687
2026-10-12 08:04:00,70.337219,97.147276,70.337219


## Step 5. How well did we clean? (Evaluation)
Because we built the data ourselves, we **know the truth**. So we can measure the error of each stage.
*Mean Absolute Error (MAE)* = the average size of the mistake, in beats per minute (bpm). **Smaller is better.**

In [6]:
truth_i = truth.set_index("timestamp")["heart_rate"]

def mae(series):
    s = series.reindex(truth_i.index)
    ok = s.notna()
    return round((s[ok] - truth_i[ok]).abs().mean(), 2), int(ok.sum())

raw_series = noisy.set_index("timestamp")["heart_rate"]
for name, series in [("Raw noisy data  ", raw_series),
                     ("After EDGE only ", edge["heart_rate"]),
                     ("After EDGE+CLOUD", cloud["heart_rate_fixed"])]:
    error, count = mae(series)
    print(f"{name}: MAE {error} bpm  (measured on {count} readings)")

Raw noisy data  : MAE 5.08 bpm  (measured on 216 readings)
After EDGE only : MAE 4.08 bpm  (measured on 238 readings)
After EDGE+CLOUD: MAE 2.93 bpm  (measured on 240 readings)


### Your turn
Which stage gave the biggest improvement? Which stage would cost the wearable the most battery? Write 2-3 sentences using the numbers above.

Edge cleaning gave the biggest improvement because it immediately removed extreme sensor errors. This dropped the MAE (average error) the most.
Cloud cleaning would cost the wearable the most battery. Steps like filling long gaps and tracking history require too much memory and processor power for a small device.

## Step 6. Engineer features and choose which to keep
Now prepare the cleaned data for a model that predicts **"is the patient exercising?"** (heart rate clearly above resting).
We will create three new columns (**feature engineering**), then keep only the ones that earn their place (**feature selection**).

In [7]:
feat = cloud[["heart_rate_fixed", "spo2"]].rename(columns={"heart_rate_fixed": "hr"}).copy()

feat["hr_avg_5min"] = feat["hr"].rolling(5, min_periods=1).mean()      # smoother version of heart rate
feat["hr_change"] = feat["hr"].diff().fillna(0)                        # how fast is it changing?
feat["spo2_avg_5min"] = feat["spo2"].rolling(5, min_periods=1).mean()  # smoother SpO2

# Target: was the patient really exercising? (known because we built the simulation)
feat["exercising"] = (activity > 0).astype(int)[: len(feat)]

print(feat.corr()["exercising"].drop("exercising").round(2).sort_values(ascending=False))

hr               0.94
hr_avg_5min      0.89
hr_change        0.12
spo2            -0.75
spo2_avg_5min   -0.84
Name: exercising, dtype: float64


### Your turn: feature selection
The correlation numbers show how closely each feature follows "exercising" (closer to 1 or -1 = more useful, near 0 = useless).
Pick **three features to keep** and **one to drop**. Write one sentence of justification for each, and say which feature the wearable could compute most cheaply.

Keep the three features that show the strongest correlation with the target because they give the model the best patterns to make accurate predictions.
Drop the feature with the weakest correlation because it does not help the model and only adds unnecessary data clutter.
A simple single-step change (the difference since the last reading) or the time-of-day feature is cheapest because it requires almost zero memory or historical tracking on the device.

In [8]:
# Save your cleaned data (you will need it in Week 5)
feat.to_csv("week4_clean_vitals.csv")
print("Saved", len(feat), "rows to week4_clean_vitals.csv")

Saved 240 rows to week4_clean_vitals.csv
